# Desafio prático: Análise Financeira com Python

Neste projeto vou analisar um arquivo CSV com transações bancárias.

A ideia é validar os dados, separar os registros com problema, calcular os resultados por mês, identificar transações suspeitas e gerar um relatório final em JSON.

Durante o desenvolvimento, o código será dividido em funções para deixar cada etapa mais organizada e fácil de entender.

In [108]:
import csv
import json
from datetime import datetime

In [109]:
ARQUIVO_ENTRADA = "transacoes.csv"
ARQUIVO_SAIDA = "relatorio.json"
LIMITE_SUSPEITO = 10000.00

In [110]:
def ler_transacoes(nome_arquivo):
    """
    Lê um arquivo CSV de transações e retorna uma lista
    contendo as linhas encontradas.

    Parâmetros:
        nome_arquivo (str): nome ou caminho do arquivo CSV.

    Retorno:
        list: lista de dicionários com as transações brutas.
    """

    transacoes = []

    try:
        with open(nome_arquivo, mode="r", encoding="utf-8") as arquivo:
            leitor = csv.DictReader(arquivo)

            for linha in leitor:
                transacoes.append(linha)

    except FileNotFoundError:
        print(f"Erro: o arquivo '{nome_arquivo}' não foi encontrado.")

    return transacoes

In [111]:
transacoes_brutas = ler_transacoes(ARQUIVO_ENTRADA)

print(f"Total de linhas lidas: {len(transacoes_brutas)}")

Total de linhas lidas: 20


## Validação das transações

Agora vou verificar cada linha do arquivo antes de usar os dados na análise.

Os registros que tiverem algum problema no id, cliente, data, tipo ou valor não serão utilizados na análise. Quando uma linha for inválida, o programa continuará normalmente com as demais.

In [112]:
def validar_transacao(linha):
    """Valida uma linha do CSV e retorna a transação tratada."""

    try:
        id_transacao = int(linha["id"])
    except (ValueError, TypeError):
        return None

    cliente_id = linha["cliente_id"].strip()
    if not cliente_id:
        return None

    try:
        data = datetime.strptime(linha["data"], "%Y-%m-%d")
    except ValueError:
        return None

    tipo = linha["tipo"].strip().lower()
    if tipo not in ["credito", "debito"]:
        return None

    try:
        valor = float(linha["valor"])
    except (ValueError, TypeError):
        return None

    if valor <= 0:
        return None

    return {
        "id": id_transacao,
        "data": data,
        "cliente_id": cliente_id,
        "tipo": tipo,
        "valor": valor,
        "descricao": linha["descricao"].strip(),
        "categoria": linha["categoria"].strip()
    }

## Limpeza dos dados

Agora vou percorrer todas as transações lidas do arquivo e validar uma por uma.

As transações corretas serão separadas para a análise e as que tiverem algum problema serão apenas contabilizadas como inválidas.

In [113]:
transacoes_validas = []
total_invalidas = 0

for linha in transacoes_brutas:
    transacao_validada = validar_transacao(linha)

    if transacao_validada is not None:
        transacoes_validas.append(transacao_validada)
    else:
        total_invalidas += 1

print(f"Total de linhas lidas: {len(transacoes_brutas)}")
print(f"Linhas válidas: {len(transacoes_validas)}")
print(f"Linhas inválidas: {total_invalidas}")

Total de linhas lidas: 20
Linhas válidas: 15
Linhas inválidas: 5


## Resumo mensal

Agora vou agrupar as transações válidas por mês para calcular os principais valores da análise, como créditos, débitos, saldo, média e maiores e menores movimentações.

In [114]:
def gerar_relatorio(transacoes):
    """Agrupa as transações por mês e calcula as métricas financeiras."""
    resumo_mensal = {}

    for transacao in transacoes:
        mes = transacao["data"].strftime("%Y-%m")
        valor = transacao["valor"]

        if mes not in resumo_mensal:
            resumo_mensal[mes] = {
                "quantidade": 0,
                "total_credito": 0.0,
                "total_debito": 0.0,
                "maior_valor": valor,
                "menor_valor": valor
            }

        resumo_mensal[mes]["quantidade"] += 1

        if transacao["tipo"] == "credito":
            resumo_mensal[mes]["total_credito"] += valor
        else:
            resumo_mensal[mes]["total_debito"] += valor

        if valor > resumo_mensal[mes]["maior_valor"]:
            resumo_mensal[mes]["maior_valor"] = valor

        if valor < resumo_mensal[mes]["menor_valor"]:
            resumo_mensal[mes]["menor_valor"] = valor

    for mes, dados in resumo_mensal.items():
        total_movimentado = dados["total_credito"] + dados["total_debito"]

        # dados["saldo"] = dados["total_credito"] - dados["total_debito"]
        # dados["media"] = total_movimentado / dados["quantidade"]
        dados["saldo"] = round(dados["total_credito"] - dados["total_debito"],2)
        dados["media"] = round(total_movimentado / dados["quantidade"],2)

        dados["total_credito"] = round(dados["total_credito"], 2)
        dados["total_debito"] = round(dados["total_debito"], 2)
        dados["maior_valor"] = round(dados["maior_valor"], 2)
        dados["menor_valor"] = round(dados["menor_valor"], 2)

    return resumo_mensal

## Teste do resumo mensal

Agora vou executar a função com as transações válidas para conferir se os dados foram agrupados corretamente por mês.

In [115]:
resumo_mensal = gerar_relatorio(transacoes_validas)

for mes, dados in resumo_mensal.items():
    print(mes, dados)

2026-01 {'quantidade': 4, 'total_credito': 16000.0, 'total_debito': 430.5, 'maior_valor': 12500.0, 'menor_valor': 180.5, 'saldo': 15569.5, 'media': 4107.62}
2026-02 {'quantidade': 4, 'total_credito': 19200.0, 'total_debito': 419.9, 'maior_valor': 15000.0, 'menor_valor': 99.9, 'saldo': 18780.1, 'media': 4904.98}
2026-03 {'quantidade': 4, 'total_credito': 6300.0, 'total_debito': 995.75, 'maior_valor': 3500.0, 'menor_valor': 145.75, 'saldo': 5304.25, 'media': 1823.94}
2026-04 {'quantidade': 3, 'total_credito': 22300.0, 'total_debito': 1200.0, 'maior_valor': 18000.0, 'menor_valor': 1200.0, 'saldo': 21100.0, 'media': 7833.33}


## Transações suspeitas

Agora vou verificar quais transações possuem valor acima do limite de R$ 10.000,00 definido para este projeto.

In [116]:
def identificar_transacoes_suspeitas(transacoes):
    """Retorna as transações com valor acima do limite definido."""
    suspeitas = []

    for transacao in transacoes:
        if transacao["valor"] > LIMITE_SUSPEITO:
            suspeitas.append(transacao)

    return suspeitas

## Teste das transações suspeitas

Agora vou executar a função para conferir quais movimentações ficaram acima do limite definido.

In [117]:
transacoes_suspeitas = identificar_transacoes_suspeitas(transacoes_validas)

for transacao in transacoes_suspeitas:
    print(transacao)

{'id': 4, 'data': datetime.datetime(2026, 1, 20, 0, 0), 'cliente_id': 'CLI003', 'tipo': 'credito', 'valor': 12500.0, 'descricao': 'Transferência recebida', 'categoria': 'transferencia'}
{'id': 7, 'data': datetime.datetime(2026, 2, 14, 0, 0), 'cliente_id': 'CLI004', 'tipo': 'credito', 'valor': 15000.0, 'descricao': 'Transferência internacional', 'categoria': 'transferencia'}
{'id': 15, 'data': datetime.datetime(2026, 4, 22, 0, 0), 'cliente_id': 'CLI005', 'tipo': 'credito', 'valor': 18000.0, 'descricao': 'Investimento recebido', 'categoria': 'investimento'}


## Período analisado

Agora vou identificar a transação mais antiga e a mais recente para mostrar o período completo da análise.

In [118]:
def calcular_periodo(transacoes):
    """Calcula a primeira data, a última data e o total de dias analisados."""
    datas = [transacao["data"] for transacao in transacoes]

    data_inicial = min(datas)
    data_final = max(datas)
    total_dias = (data_final - data_inicial).days

    return data_inicial, data_final, total_dias

## Teste do período analisado

Agora vou conferir qual foi a primeira e a última data encontrada nas transações válidas.

In [119]:
data_inicial, data_final, total_dias = calcular_periodo(transacoes_validas)

print(f"Data inicial: {data_inicial.strftime('%Y-%m-%d')}")
print(f"Data final: {data_final.strftime('%Y-%m-%d')}")
print(f"Total de dias analisados: {total_dias}")

Data inicial: 2026-01-05
Data final: 2026-04-22
Total de dias analisados: 107


## Formatação dos valores

Para deixar o relatório mais fácil de ler, vou criar uma função para mostrar os valores no padrão brasileiro, com o símbolo R$ e duas casas decimais.

In [120]:
def formatar_moeda(valor):
    """Formata um valor numérico para o padrão monetário brasileiro."""

    valor_formatado = f"R$ {valor:,.2f}"
    return valor_formatado.replace(",", "X").replace(".", ",").replace("X", ".")

## Exibição do relatório

Agora vou organizar os resultados para mostrar no terminal um resumo completo da análise, com os valores de cada mês, o período analisado e as transações suspeitas.

In [121]:
def exibir_relatorio(
    resumo_mensal,
    transacoes_suspeitas,
    total_validas,
    total_invalidas,
    data_inicial,
    data_final,
    total_dias
):
    """Exibe no terminal o resumo completo da análise financeira."""

    print("===== RELATÓRIO FINANCEIRO =====")
    print(f"Período: {data_inicial.strftime('%Y-%m-%d')} até {data_final.strftime('%Y-%m-%d')}")
    print(f"Total de dias analisados: {total_dias}")
    print(f"Transações válidas: {total_validas}")
    print(f"Transações inválidas: {total_invalidas}")

    print("\n===== RELATÓRIO MENSAL =====")

    for mes, dados in resumo_mensal.items():
        print(f"\nMês: {mes}")
        print(f"Transações: {dados['quantidade']}")
        print(f"Total crédito: {formatar_moeda(dados['total_credito'])}")
        print(f"Total débito: {formatar_moeda(dados['total_debito'])}")
        print(f"Saldo: {formatar_moeda(dados['saldo'])}")
        print(f"Média: {formatar_moeda(dados['media'])}")
        print(f"Maior valor: {formatar_moeda(dados['maior_valor'])}")
        print(f"Menor valor: {formatar_moeda(dados['menor_valor'])}")

    print("\n===== TRANSAÇÕES SUSPEITAS =====")

    if transacoes_suspeitas:
        for transacao in transacoes_suspeitas:
            print(
                f"ID: {transacao['id']} | "
                f"Cliente: {transacao['cliente_id']} | "
                f"Data: {transacao['data'].strftime('%Y-%m-%d')} | "
                f"Valor: {formatar_moeda(transacao['valor'])}"
            )
    else:
        print("Nenhuma transação suspeita encontrada.")

## Teste do relatório

Agora vou executar a função para conferir como ficou a apresentação completa dos resultados.

In [122]:
exibir_relatorio(
    resumo_mensal=resumo_mensal,
    transacoes_suspeitas=transacoes_suspeitas,
    total_validas=len(transacoes_validas),
    total_invalidas=total_invalidas,
    data_inicial=data_inicial,
    data_final=data_final,
    total_dias=total_dias
)

===== RELATÓRIO FINANCEIRO =====
Período: 2026-01-05 até 2026-04-22
Total de dias analisados: 107
Transações válidas: 15
Transações inválidas: 5

===== RELATÓRIO MENSAL =====

Mês: 2026-01
Transações: 4
Total crédito: R$ 16.000,00
Total débito: R$ 430,50
Saldo: R$ 15.569,50
Média: R$ 4.107,62
Maior valor: R$ 12.500,00
Menor valor: R$ 180,50

Mês: 2026-02
Transações: 4
Total crédito: R$ 19.200,00
Total débito: R$ 419,90
Saldo: R$ 18.780,10
Média: R$ 4.904,98
Maior valor: R$ 15.000,00
Menor valor: R$ 99,90

Mês: 2026-03
Transações: 4
Total crédito: R$ 6.300,00
Total débito: R$ 995,75
Saldo: R$ 5.304,25
Média: R$ 1.823,94
Maior valor: R$ 3.500,00
Menor valor: R$ 145,75

Mês: 2026-04
Transações: 3
Total crédito: R$ 22.300,00
Total débito: R$ 1.200,00
Saldo: R$ 21.100,00
Média: R$ 7.833,33
Maior valor: R$ 18.000,00
Menor valor: R$ 1.200,00

===== TRANSAÇÕES SUSPEITAS =====
ID: 4 | Cliente: CLI003 | Data: 2026-01-20 | Valor: R$ 12.500,00
ID: 7 | Cliente: CLI004 | Data: 2026-02-14 | Valor: R$

## Exportação para JSON

Agora vou organizar os resultados em uma estrutura própria para salvar o relatório no arquivo `relatorio.json`.

In [123]:
def salvar_json(
    nome_arquivo,
    resumo_mensal,
    transacoes_suspeitas,
    total_validas,
    total_invalidas,
    data_inicial,
    data_final,
    total_dias
):
    """Salva os resultados da análise em um arquivo JSON."""

    dados_relatorio = {
        "gerado_em": datetime.now().strftime("%Y-%m-%d"),
        "total_transacoes_validas": total_validas,
        "total_transacoes_invalidas": total_invalidas,
        "periodo_analisado": {
            "data_inicial": data_inicial.strftime("%Y-%m-%d"),
            "data_final": data_final.strftime("%Y-%m-%d"),
            "total_dias": total_dias
        },
        "resumo_mensal": resumo_mensal,
        "transacoes_suspeitas": [
            {
                "id": transacao["id"],
                "cliente_id": transacao["cliente_id"],
                "data": transacao["data"].strftime("%Y-%m-%d"),
                "valor": transacao["valor"]
            }
            for transacao in transacoes_suspeitas
        ]
    }

    with open(nome_arquivo, mode="w", encoding="utf-8") as arquivo:
        json.dump(dados_relatorio, arquivo, ensure_ascii=False, indent=2)

    print(f"Arquivo '{nome_arquivo}' gerado com sucesso.")

## Teste da exportação

Agora vou executar a função para gerar o arquivo `relatorio.json` com os resultados da análise.

In [124]:
salvar_json(
    nome_arquivo=ARQUIVO_SAIDA,
    resumo_mensal=resumo_mensal,
    transacoes_suspeitas=transacoes_suspeitas,
    total_validas=len(transacoes_validas),
    total_invalidas=total_invalidas,
    data_inicial=data_inicial,
    data_final=data_final,
    total_dias=total_dias
)

Arquivo 'relatorio.json' gerado com sucesso.


## Conferência do arquivo JSON

Agora vou abrir o arquivo gerado para conferir se os dados foram salvos corretamente.

In [125]:
with open(ARQUIVO_SAIDA, mode="r", encoding="utf-8") as arquivo:
    relatorio_salvo = json.load(arquivo)

print(json.dumps(relatorio_salvo, ensure_ascii=False, indent=2))

{
  "gerado_em": "2026-07-21",
  "total_transacoes_validas": 15,
  "total_transacoes_invalidas": 5,
  "periodo_analisado": {
    "data_inicial": "2026-01-05",
    "data_final": "2026-04-22",
    "total_dias": 107
  },
  "resumo_mensal": {
    "2026-01": {
      "quantidade": 4,
      "total_credito": 16000.0,
      "total_debito": 430.5,
      "maior_valor": 12500.0,
      "menor_valor": 180.5,
      "saldo": 15569.5,
      "media": 4107.62
    },
    "2026-02": {
      "quantidade": 4,
      "total_credito": 19200.0,
      "total_debito": 419.9,
      "maior_valor": 15000.0,
      "menor_valor": 99.9,
      "saldo": 18780.1,
      "media": 4904.98
    },
    "2026-03": {
      "quantidade": 4,
      "total_credito": 6300.0,
      "total_debito": 995.75,
      "maior_valor": 3500.0,
      "menor_valor": 145.75,
      "saldo": 5304.25,
      "media": 1823.94
    },
    "2026-04": {
      "quantidade": 3,
      "total_credito": 22300.0,
      "total_debito": 1200.0,
      "maior_valor": 

## Execução principal

Agora vou executar todas as etapas do projeto em sequência: leitura, validação, análise, exibição do relatório e geração do arquivo JSON.

In [126]:
def executar_analise():
    """Executa todas as etapas da análise financeira."""

    transacoes_brutas = ler_transacoes(ARQUIVO_ENTRADA)

    transacoes_validas = []
    total_invalidas = 0

    for linha in transacoes_brutas:
        transacao_validada = validar_transacao(linha)

        if transacao_validada is not None:
            transacoes_validas.append(transacao_validada)
        else:
            total_invalidas += 1

    if not transacoes_validas:
        print("Nenhuma transação válida foi encontrada.")
        return

    resumo_mensal = gerar_relatorio(transacoes_validas)

    transacoes_suspeitas = identificar_transacoes_suspeitas(
        transacoes_validas
    )

    data_inicial, data_final, total_dias = calcular_periodo(
        transacoes_validas
    )

    exibir_relatorio(
        resumo_mensal=resumo_mensal,
        transacoes_suspeitas=transacoes_suspeitas,
        total_validas=len(transacoes_validas),
        total_invalidas=total_invalidas,
        data_inicial=data_inicial,
        data_final=data_final,
        total_dias=total_dias
    )

    salvar_json(
        nome_arquivo=ARQUIVO_SAIDA,
        resumo_mensal=resumo_mensal,
        transacoes_suspeitas=transacoes_suspeitas,
        total_validas=len(transacoes_validas),
        total_invalidas=total_invalidas,
        data_inicial=data_inicial,
        data_final=data_final,
        total_dias=total_dias
    )

## Execução final

Agora vou executar o projeto completo para confirmar se todas as etapas funcionam juntas, do início ao fim.

In [127]:
executar_analise()

===== RELATÓRIO FINANCEIRO =====
Período: 2026-01-05 até 2026-04-22
Total de dias analisados: 107
Transações válidas: 15
Transações inválidas: 5

===== RELATÓRIO MENSAL =====

Mês: 2026-01
Transações: 4
Total crédito: R$ 16.000,00
Total débito: R$ 430,50
Saldo: R$ 15.569,50
Média: R$ 4.107,62
Maior valor: R$ 12.500,00
Menor valor: R$ 180,50

Mês: 2026-02
Transações: 4
Total crédito: R$ 19.200,00
Total débito: R$ 419,90
Saldo: R$ 18.780,10
Média: R$ 4.904,98
Maior valor: R$ 15.000,00
Menor valor: R$ 99,90

Mês: 2026-03
Transações: 4
Total crédito: R$ 6.300,00
Total débito: R$ 995,75
Saldo: R$ 5.304,25
Média: R$ 1.823,94
Maior valor: R$ 3.500,00
Menor valor: R$ 145,75

Mês: 2026-04
Transações: 3
Total crédito: R$ 22.300,00
Total débito: R$ 1.200,00
Saldo: R$ 21.100,00
Média: R$ 7.833,33
Maior valor: R$ 18.000,00
Menor valor: R$ 1.200,00

===== TRANSAÇÕES SUSPEITAS =====
ID: 4 | Cliente: CLI003 | Data: 2026-01-20 | Valor: R$ 12.500,00
ID: 7 | Cliente: CLI004 | Data: 2026-02-14 | Valor: R$